# Fase 1 — Treino e serialização do modelo (MNIST → MLP)

Este notebook prepara o modelo que a API vai servir. Estrutura:

1. Objetivo e origem do modelo
2. Carga dos dados
3. Divisão estratificada em treino, validação e teste
4. Normalização dos pixels
5. Treino do MLP
6. Avaliação no conjunto de teste
7. Serialização com joblib
8. Relatório de métricas

> **Como usar este esqueleto:** os textos em itálico são perguntas-guia.
> Substitua cada um pela sua explicação, com as suas palavras, e escreva o código
> nas células vazias. Depois de pronto, não deve sobrar nenhum texto de orientação.



## 1. Objetivo e origem do modelo

Este notebook treina o modelo que será servido pela API e grava os arquivos que ela precisa: o modelo serializado em `models/mnist_mlp.joblib` e o relatório de métricas em `reports/metrics.json`.

O modelo é um MLP (Perceptron Multicamadas), uma rede neural, reaproveitado do projeto mnist-modulo2 (classificação de dígitos MNIST). Foi escolhido porque, naquele projeto, teve uma acurácia de cerca de 97,8% no teste, a maior entre os 3 modelos comparados, a predição mais rápida: ~0,08 s para 14.000 imagens (Random Forest 0,87 s e KNN 15,3 s) e gera um arquivo pequeno.

Hiperparâmetros usados (já ajustados e validados no projeto anterior):

- `hidden_layer_sizes=(128, 64)`: duas camadas ocultas, com 128 e 64 neurônios. Controla a capacidade da rede de aprender padrões;
- `alpha=0.001`: força da regularização L2, uma “punição” a pesos muito grandes que reduz o overfitting;
- `early_stopping=True`: a rede separa parte do treino para se auto-avaliar e para quando deixa de melhorar;
- `random_state=42`: fixa a semente aleatória, para o resultado ser reproduzível.

### 1.1 Importações

In [ ]:
# Importando as bibliotecas necessárias: numpy, fetch_openml e train_test_split

import numpy as np

from sklearn.datasets import fetch_openml


## 2. Carga dos dados

O `mnist_784` tem 70000 imagens de dígitos manuscritos, cada uma com 784 pixels (imagens de 28 × 28 achatadas em uma linha). Cada coluna é a intensidade de um pixel numa posição fixa da imagem e vai de 0 (preto) a 255 (branco).

Os rótulos (`y`) chegam como texto, porque o OpenML guarda o alvo como categoria. Por isso converti para inteiro (uint8) com `astype`.


In [ ]:
# Carregando o MNIST (as_frame=False), convertemdo y para inteiro e imprimindo os formatos e os tipos
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
X, y = mnist.data, mnist.target
y = y.astype(np.uint8)  
print(X.shape, y.shape, X.dtype, X.min(), X.max())


(70000, 784) (70000,) int64 0 255



## 3. Divisão estratificada em treino, validação e teste

Dividi os dados em 70% treino, 10% validação, 20% teste. A divisão foi feita em 2 etapas: primeiro separei 20% para o teste e, dos 80% restantes, separei uma fração de 0,125 para a validação. Essa segunda fração é diferente da proporção final desejada porque é calculada sobre os 80% (10% do total ÷ 80% = 0,125.)

Usei `stratify` para manter a mesma proporção de cada dígito em todos os conjuntos. Isso importa neste dataset porque evita que, por azar do sorteio, um dígito fique sub-representado no treino ou no teste.

- **Treino:** onde o modelo aprende os padrões;
- **Validação:** ajusta e compara configurações sem gastar o teste (usada no projeto anterior)
- **Teste:** é a medida final e honesta do desempenho; só será usado no final, na avaliação (seção 6)

Fixei `random_state=42`, para a divisão ser reproduzível (mesma partição para quem rodar)

Resultado: 49000 amostras de treino, 7000 de validação e 14000 de teste.

In [5]:
# Divida X e y em treino (70%), validação (10%) e teste (20%), com stratify e random_state fixo
# Imprima os formatos dos seis arrays
from sklearn.model_selection import train_test_split


X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.125, stratify=y_temp, random_state=42
)

print("Treino:   ", X_train.shape, y_train.shape)
print("Validação:", X_val.shape, y_val.shape)
print("Teste:    ", X_test.shape, y_test.shape)

Treino:    (49000, 784) (49000,)
Validação: (7000, 784) (7000,)
Teste:     (14000, 784) (14000,)


## 4. Normalização dos pixels

Os pixels vão de 0 a 255 e eu os levei para a escala de 0.0 a 1.0, dividindo por 255.0.

Normalizar ajuda o MLP porque valores pequenos e na mesma escala deixam a otimização por gradiente mais estável e rápida; valores grandes a deixam lenta e instável.

Dividir por uma constante não vaza informação do teste para o treino porque 255 é o valor máximo teórico de um pixel de 8 bits, conhecido de antemão, e não é calculado a partir dos dados. Já um scaler ajustado com `.fit()` calcula estatísticas dos dados (média, desvio), então precisaria ser ajustado só no treino e depois aplicado nos outros conjuntos.

**Contrato com a API:** a API precisa dividir os pixels por 255.0 antes de prever, exatamente como no treino. Se isso não acontecer, o modelo recebe valores numa escala diferente da do treino e as previsões ficam erradas (isso se chama train/serve skew). Esse valor será registrado em `reports/metrics.json` (na seção 8).

In [6]:
# Normalização: divide cada pixel por 255.0 para que os valores fiquem entre 0 e 1.
X_train = X_train / 255.0
X_val   = X_val / 255.0
X_test  = X_test / 255.0
print("Min:", X_train.min(), "| Max:", X_train.max())
print("Min:", X_val.min(), "| Max:", X_val.max())
print("Min:", X_test.min(), "| Max:", X_test.max())


Min: 0.0 | Max: 1.0
Min: 0.0 | Max: 1.0
Min: 0.0 | Max: 1.0


## 5. Treino do MLP

*(próxima etapa)*


## 6. Avaliação no conjunto de teste

*(próxima etapa)*


## 7. Serialização com joblib

*(próxima etapa)*


## 8. Relatório de métricas

*(próxima etapa)*
